# 07 — Retail Planner: Knowledge Base Engineering & RAG

## 1. Business Objectives and Knowledge Architecture

### 1.1 Business context

Our retail demand forecasting platform produces daily sales forecasts for 1,782 store-family series over a 14-day horizon.

The platform already includes data preparation, temporal forecasting, model evaluation, AWS batch inference, model registration and guarded MLOps workflows.

However, numerical forecasts alone do not provide the explanations needed for informed retail planning decisions.

A retail planner may need to understand:

- Which forecasting model is currently selected, and why.
- How forecast accuracy should be interpreted across stores and product families.
- Whether a forecast exhibits systematic overestimation or underestimation.
- Which data limitations may affect forecast reliability.
- Why certain experimental modeling approaches were rejected.
- Which assumptions should be considered before using a forecast for inventory decisions.

The objective is to make reliable, domain-specific information accessible to an AI-powered forecasting assistant.

### 1.2 The knowledge engineering problem

In a realistic enterprise environment, useful information is distributed across heterogeneous sources:

- Machine-readable model artifacts and configuration files.
- Technical documentation and data contracts.
- Research notebooks and experimental reports.
- Business specifications and operational procedures.
- Historical decisions and model evaluation results.

These sources differ in authority, structure, completeness and freshness.

A research notebook may contain several contradictory experimental conclusions before reaching a final decision. A model contract may accurately describe one model version but provide little business interpretation. A README may summarize the system without containing every technical detail.

Therefore, **document availability does not imply document suitability for retrieval**.

The challenge is to transform heterogeneous project information into a reliable, maintainable knowledge layer.

### 1.3 Knowledge Base vs Retrieval-Augmented Generation

A **Knowledge Base (KB)** is an organized collection of information maintained for a defined purpose.

Its quality depends on source selection, factual correctness, provenance, structure, versioning and maintenance.

**Retrieval-Augmented Generation (RAG)** is a method that retrieves external information and provides it as context to a language model during response generation.

The Knowledge Base defines what reliable information is available.

Retrieval determines which information is relevant to a particular question.

Generation uses that information to construct a grounded answer.

Importantly, not every information request requires semantic search or a vector database.

A factual query about a model's WAPE may be better answered through deterministic access to a validated JSON contract, while a question about the business implications of forecast bias may benefit from retrieving a curated explanatory document.

### 1.4 Target knowledge architecture

We distinguish three information categories.

| Category | Purpose | Preferred access |
|---|---|---|
| **Structured facts** | Validated model configurations, metrics, versions and contracts | Deterministic lookup |
| **Curated knowledge** | Business explanations, model decisions, assumptions and limitations | Document retrieval |
| **Operational data** | Store-family forecasts, historical observations and monitoring results | Dedicated tools |

The Knowledge Base developed in this notebook will primarily address structured facts and curated knowledge.

Operational tools will be implemented as part of the later Agentic AI architecture.

### 1.5 Objectives of this notebook

This notebook develops the knowledge engineering and retrieval foundations of the Retail Forecast Investigation Assistant.

The objectives are to:

1. Translate business questions into explicit knowledge requirements.
2. Discover and classify relevant project information sources.
3. Assess source authority, completeness and potential inconsistencies.
4. Extract, validate and structure the information needed by the assistant.
5. Build a small, versioned Knowledge Base with stable identifiers and provenance.
6. Implement and compare appropriate retrieval strategies.
7. Evaluate retrieval relevance, evidence coverage and grounded question answering.
8. Select a maintainable knowledge-access interface for downstream GenAI and agentic components.

The goal is not to index every available document.

It is to construct the smallest reliable knowledge layer that satisfies the intended business requirements.

### 1.6 Engineering principles

- **Business-first:** define information requirements before choosing retrieval technology.
- **Source-of-truth awareness:** distinguish authoritative artifacts from research history and secondary summaries.
- **Explicit validation:** do not promote an experimental finding into authoritative knowledge without appropriate evidence.
- **Stable provenance:** use versioned source references and semantic identifiers independent of notebook cell positions.
- **Minimal duplication:** read quantitative facts from validated artifacts whenever possible.
- **Maintainability:** support updates when models, contracts or business rules change.
- **Access control:** do not expose information beyond its authorized audience.
- **Measurable quality:** evaluate retrieval against questions and evidence requirements rather than relying on plausible-looking answers.
- **Proportional complexity:** introduce embeddings, vector search or reranking only when simpler approaches are insufficient.

#### Business accessibility and user adoption

The primary user is a retail planner, not a machine learning specialist.

The assistant must communicate in a way that supports business understanding and decision-making without requiring prior knowledge of statistics, machine learning or software engineering.

**Default communication policy:**

- Answer the user's actual question directly, using clear and concise business language.
- Prioritize what the result means for the planner, rather than explaining the underlying statistical methodology.
- Avoid technical jargon unless the user introduces it or explicitly requests technical details.
- Do not overwhelm users with caveats, secondary metrics or explanations they did not ask for.
- Mention limitations or uncertainty when they materially affect the reliability or interpretation of the answer.
- Offer deeper explanations when useful, without imposing them by default.
- Adapt the level of technical detail to the user's language, expertise and follow-up questions.

**The objective is not to teach forecasting to retail planners. It is to help them make better decisions with minimal cognitive effort.**

User adoption is therefore a product-quality requirement, alongside factual reliability, retrieval relevance and system performance.

### 1.7 Experimental roadmap

| Stage | Objective | Deliverable |
|---|---|---|
| **1. Business Objectives & Architecture** | Define information needs and access patterns | Knowledge requirements |
| **2. Source Discovery & Assessment** | Identify authoritative, relevant and potentially conflicting information | Source inventory and classification |
| **3. Knowledge Curation & Validation** | Extract and validate reliable facts and explanations | Reviewed knowledge items |
| **4. Knowledge Base Construction** | Structure, version and store curated knowledge | Reproducible knowledge artifacts |
| **5. Retrieval Implementation** | Compare appropriate access strategies | Retrieval baselines |
| **6. Retrieval & Grounding Evaluation** | Measure relevance, completeness and factual support | Evaluation results |
| **7. Conclusions & GenAI Handoff** | Define the selected knowledge-access interface | Reusable components for Notebook 08 |

The next notebook will focus on GenAI model behavior, prompting, fine-tuning and LLMOps.

A subsequent notebook will integrate these capabilities into a tool-using AI agent, including Model Context Protocol (MCP) experiments and AgentOps.

## 2. Source Discovery and Assessment

### 2.1 From business questions to information requirements

Knowledge engineering should begin with the questions the system must answer, not with the documents available for indexing.

For each business question, we identify:

1. **Required evidence:** what information is necessary to answer correctly?
2. **Expected source:** where should that information originate?
3. **Authority:** which artifact or process validates the information?
4. **Access method:** should the information be retrieved as a document, read as structured data, or obtained through an operational tool?

We begin with a small representative set of forecasting questions.

| ID | Retail planner question | Required evidence | Expected source |
|---|---|---|---|
| K01 | Which forecasting system are we using, and why? | Current champion, model-selection rationale, training policy | Validated model contract and model card |
| K02 | How accurate are our forecasts? | Historical test results, reference benchmark, evaluation limitations | Evaluation artifacts and model card |
| K03 | Can I trust these forecasts for every store and product? | Performance heterogeneity, aggregation effects, known limitations | Evaluation analysis and curated methodology |
| K04 | Does the system tend to predict too many or too few sales? | Directional forecast bias and its inventory-planning implications | Evaluation metrics and business methodology |
| K05 | Does the system take promotions into account? | Current feature scope, experimental findings, final model-selection decision | Feature contract and promotion-analysis documentation |
| K06 | Does low sales volume always mean customers didn't want the product? | Observed sales versus unconstrained demand, missing stock availability | Dataset documentation and business glossary |
| K07 | What should we expect to sell in my store next week? | Store-family forecasts, prediction dates and model version | Operational forecast artifact or dedicated tool |

The questions deliberately use retail-planning language rather than technical terminology.

Technical concepts such as WAPE, normalized bias or causal uplift belong in the supporting evidence and explanatory knowledge, not necessarily in the user's question.

K07 requires operational data access and will be implemented in the agent layer. It is retained here to make the boundary between knowledge retrieval and operational tools explicit.

### 2.2 Source discovery strategy

We first inventory the project's version-controlled information sources.

We distinguish:

- **Structured artifacts:** potential sources of authoritative machine-readable facts.
- **Project documentation:** specifications, architecture and operational descriptions.
- **Research notebooks:** experimental history and supporting evidence.
- **Data transformation definitions:** technical references for data semantics and lineage.

Classification at this stage is preliminary.

A file's name, location or format is not sufficient to establish its correctness or authority.

We will inspect candidate sources against the information requirements before deciding what belongs in the Knowledge Base.

### 2.3 Reproducibility and provenance

Source discovery should produce a traceable record rather than relying on undocumented terminal exploration.

The inventory records repository-relative paths and the Git revision used during inspection.

However, a Git revision alone identifies committed content; uncommitted local changes must be treated separately.

Only reviewed and validated information will be promoted into curated knowledge artifacts.

Raw research notebooks will remain supporting evidence, not automatic retrieval documents.

In [1]:
from pathlib import Path
import subprocess

import pandas as pd

# Locate the Git repository.
current_dir = Path.cwd().resolve()

repo_result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"],
    cwd=current_dir,
    capture_output=True,
    text=True,
    check=True,
)

project_root = Path(repo_result.stdout.strip())

revision = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=project_root,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

tracked_result = subprocess.run(
    ["git", "ls-files"],
    cwd=project_root,
    capture_output=True,
    text=True,
    check=True,
)

tracked_paths = tracked_result.stdout.splitlines()

def classify_source(path):
    """Classify potential information sources by repository location."""
    if path.startswith("artifacts/") and path.endswith(".json"):
        return "Structured artifact"
    if path.startswith("docs/") or path == "README.md":
        return "Project documentation"
    if path.startswith("notebooks/") and path.endswith(".ipynb"):
        return "Research notebook"
    if path.startswith("dbt/") and path.endswith(
        (".sql", ".yml", ".yaml")
    ):
        return "Data transformation reference"
    return None

inventory_rows = []

for path in tracked_paths:
    category = classify_source(path)

    if category is not None:
        inventory_rows.append({
            "source_path": path,
            "category": category,
            "extension": Path(path).suffix,
        })

source_inventory = pd.DataFrame(
    inventory_rows,
    columns=["source_path", "category", "extension"],
)

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=project_root,
    capture_output=True,
    text=True,
    check=True,
)

print("SOURCE DISCOVERY")
print("Repository:", project_root.name)
print("Git revision:", revision[:12])
print("Uncommitted changes present:", bool(status_result.stdout.strip()))
print("Tracked files:", len(tracked_paths))
print("Potential information sources:", len(source_inventory))

print("\nSOURCE CATEGORIES")
category_summary = (
    source_inventory.groupby("category")
    .size()
    .reset_index(name="file_count")
    .sort_values("category")
)
display(category_summary.style.hide(axis="index"))

print("\nSTRUCTURED ARTIFACTS AND PROJECT DOCUMENTATION")
priority_sources = source_inventory.loc[
    source_inventory["category"].isin([
        "Structured artifact",
        "Project documentation",
    ])
]
display(priority_sources.head(40).style.hide(axis="index"))

SOURCE DISCOVERY
Repository: retail-demand-forecasting-platform-aws
Git revision: d2d5a9c20773
Uncommitted changes present: True
Tracked files: 220
Potential information sources: 109

SOURCE CATEGORIES


category,file_count
Data transformation reference,12
Project documentation,73
Research notebook,6
Structured artifact,18



STRUCTURED ARTIFACTS AND PROJECT DOCUMENTATION


source_path,category,extension
README.md,Project documentation,.md
artifacts/aws/continuous_training/last_successful_ct.json,Structured artifact,.json
artifacts/aws/continuous_training/readiness_v1.json,Structured artifact,.json
artifacts/aws/sagemaker_batch_inference/v1/cloud/inference_report.json,Structured artifact,.json
artifacts/aws/sagemaker_batch_inference/v1/features.manifest.json,Structured artifact,.json
artifacts/aws/sagemaker_batch_inference/v1/local_parity/inference_report.json,Structured artifact,.json
artifacts/aws/sagemaker_processing/bootstrap_v1/job_summary.json,Structured artifact,.json
artifacts/aws/sagemaker_processing/bootstrap_v1/validation_report_v1.json,Structured artifact,.json
artifacts/aws/sagemaker_processing/evaluation_v1/holdout_actuals_manifest.json,Structured artifact,.json
artifacts/aws/sagemaker_processing/evaluation_v1/local_check/evaluation_report.json,Structured artifact,.json


In [2]:
import pandas as pd

# Inspect file types within the initial source inventory.
source_type_summary = (
    source_inventory
    .groupby(["category", "extension"])
    .size()
    .reset_index(name="file_count")
    .sort_values(["category", "file_count"], ascending=[True, False])
)

print("SOURCE TYPES")
display(source_type_summary.style.hide(axis="index"))

# Identify textual documentation that may contain reusable knowledge.
documentation_candidates = source_inventory.loc[
    (source_inventory["category"] == "Project documentation")
    & (source_inventory["extension"].isin([".md", ".rst", ".txt"]))
].sort_values("source_path")

print("\nTEXTUAL DOCUMENTATION CANDIDATES")
print("Files:", len(documentation_candidates))
display(
    documentation_candidates[["source_path", "extension"]]
    .style.hide(axis="index")
)

# Identify structured artifacts for subsequent authority assessment.
structured_candidates = source_inventory.loc[
    source_inventory["category"] == "Structured artifact"
].sort_values("source_path")

print("\nSTRUCTURED ARTIFACT CANDIDATES")
print("Files:", len(structured_candidates))
display(
    structured_candidates[["source_path", "extension"]]
    .style.hide(axis="index")
)

SOURCE TYPES


category,extension,file_count
Data transformation reference,.sql,8
Data transformation reference,.yml,4
Project documentation,.png,66
Project documentation,.csv,6
Project documentation,.md,1
Research notebook,.ipynb,6
Structured artifact,.json,18



TEXTUAL DOCUMENTATION CANDIDATES
Files: 1


source_path,extension
README.md,.md



STRUCTURED ARTIFACT CANDIDATES
Files: 18


source_path,extension
artifacts/aws/continuous_training/last_successful_ct.json,.json
artifacts/aws/continuous_training/readiness_v1.json,.json
artifacts/aws/sagemaker_batch_inference/v1/cloud/inference_report.json,.json
artifacts/aws/sagemaker_batch_inference/v1/features.manifest.json,.json
artifacts/aws/sagemaker_batch_inference/v1/local_parity/inference_report.json,.json
artifacts/aws/sagemaker_processing/bootstrap_v1/job_summary.json,.json
artifacts/aws/sagemaker_processing/bootstrap_v1/validation_report_v1.json,.json
artifacts/aws/sagemaker_processing/evaluation_v1/holdout_actuals_manifest.json,.json
artifacts/aws/sagemaker_processing/evaluation_v1/local_check/evaluation_report.json,.json
artifacts/aws/sagemaker_registry/challenger_ct_v2/registry_record.json,.json


### 2.4 Source authority and cross-validation

Source discovery identified several machine-readable artifacts related to the forecasting model.

Before incorporating their information into the Knowledge Base, we must verify that the sources are consistent and understand what each source actually establishes.

We begin with two business questions:

- **K01:** Which forecasting system are we using, and why?
- **K02:** How accurate are our forecasts?

Three existing artifacts provide complementary evidence:

| Source | Primary responsibility |
|---|---|
| XGBoost V1 model contract | Selected model configuration, input features and final evaluation metrics |
| Cloud evaluation report | Independent cloud reproduction of final holdout metrics |
| SageMaker Registry record | Model registration metadata, artifact integrity and reproduction audit |

These sources do not have identical authority.

The model contract describes a specific frozen model. The cloud evaluation report validates its historical forecasting results. The registry record documents registration and audit information, but should not be interpreted as proof of an active production deployment.

We first check agreement on the model artifact, evaluation period and key performance metrics.

This prevents the future Knowledge Base from combining inconsistent model versions or presenting an experimental result as an operational fact.

In [3]:
import json
import math
import pandas as pd

source_paths = {
    "contract": "artifacts/global_xgboost/point_forecast_model_contract_v1.json",
    "evaluation": "artifacts/aws/sagemaker_registry/champion_v1/evaluation_report_cloud.json",
    "registry": "artifacts/aws/sagemaker_registry/champion_v1/registry_record.json",
}

sources = {
    name: json.loads((project_root / path).read_text(encoding="utf-8"))
    for name, path in source_paths.items()
}

contract = sources["contract"]
evaluation = sources["evaluation"]
registry = sources["registry"]

def metrics_agree(a, b):
    return math.isclose(a, b, rel_tol=0, abs_tol=1e-8)

validation_checks = [
    {
        "check": "Same model artifact SHA256",
        "passed": (
            contract["artifact"]["package_sha256"]
            == registry["model_sha256"]
        ),
    },
    {
        "check": "Same holdout period",
        "passed": (
            contract["final_holdout_start"] == evaluation["holdout_start"]
            and contract["final_holdout_end"] == evaluation["holdout_end"]
        ),
    },
    {
        "check": "Consistent holdout WAPE",
        "passed": metrics_agree(
            contract["final_holdout_metrics"]["wape"],
            evaluation["portfolio"]["wape"],
        ),
    },
    {
        "check": "Consistent normalized bias",
        "passed": metrics_agree(
            contract["final_holdout_metrics"]["normalized_bias"],
            evaluation["portfolio"]["normalized_bias"],
        ),
    },
    {
        "check": "Cloud evaluation passed",
        "passed": evaluation["status"] == "PASSED",
    },
    {
        "check": "Registry audit passed",
        "passed": registry["audit"]["status"] == "PASSED",
    },
]

validation_results = pd.DataFrame(validation_checks)

print("MODEL SOURCE CROSS-VALIDATION")
display(validation_results.style.hide(axis="index"))

print("\nMODEL CONTEXT")
print("Champion ID:", contract["champion_id"])
print("Model version:", contract["artifact"]["model_version"])
print("Evaluation period:", evaluation["holdout_start"], "to", evaluation["holdout_end"])
print("Cloud WAPE:", f'{evaluation["portfolio"]["wape"]:.2%}')
print("Registry audit type:", registry["audit"]["gate_type"])

assert validation_results["passed"].all(), "Source inconsistency detected"

MODEL SOURCE CROSS-VALIDATION


check,passed
Same model artifact SHA256,True
Same holdout period,True
Consistent holdout WAPE,True
Consistent normalized bias,True
Cloud evaluation passed,True
Registry audit passed,True



MODEL CONTEXT
Champion ID: candidate_27
Model version: point_forecast_v1
Evaluation period: 2017-08-02 to 2017-08-15
Cloud WAPE: 18.77%
Registry audit type: frozen_model_reproduction_audit_not_model_selection


### 2.5 Initial Findings and Evidence Gaps

The first source-validation step passed all six consistency checks.

The frozen XGBoost V1 model contract, cloud evaluation report and SageMaker Registry audit agree on the model artifact and final holdout metrics.

This establishes a reliable factual foundation for the selected model version.

However, **technical consistency does not automatically establish business relevance or answer completeness**.

For our first two knowledge requirements:

| Requirement | Validated evidence | Remaining gap |
|---|---|---|
| K01 — Which forecasting system are we using, and why? | Model identity, training policy, feature schema and artifact integrity | Model-selection rationale and operational limitations |
| K02 — How accurate are our forecasts? | Reproduced final holdout metrics | Baseline comparison, historical context and interpretation |

An additional distinction concerns continuous training.

The latest recorded CT checkpoint reached challenger registration with manual approval pending. The challenger was not promoted, and no new out-of-sample performance was available.

Therefore, successful CT execution must not be confused with demonstrated challenger superiority or a change in the selected forecasting model.

#### Next investigation

We will inspect the existing forecasting evaluation summaries to establish the comparison baseline and historical context.

Only evidence directly relevant to the identified knowledge gaps will be examined.

The objective is to produce a concise, validated explanation suitable for a retail planner, while preserving the technical details and provenance needed for auditability.

In [4]:
import pandas as pd

evaluation_sources = {
    "Final holdout model comparison": (
        "docs/assets/global_forecasting/"
        "final_holdout_model_comparison.csv"
    ),
    "Historical evaluation context": (
        "docs/assets/global_forecasting/"
        "final_holdout_historical_context.csv"
    ),
}

evaluation_tables = {}

for source_name, relative_path in evaluation_sources.items():
    file_path = project_root / relative_path

    if not file_path.is_file():
        raise FileNotFoundError(f"Missing evidence source: {file_path}")

    table = pd.read_csv(file_path)
    evaluation_tables[source_name] = table

    print(f"\nSOURCE: {source_name}")
    print("Path:", relative_path)
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())

    display(table.head(10).style.hide(axis="index"))


SOURCE: Final holdout model comparison
Path: docs/assets/global_forecasting/final_holdout_model_comparison.csv
Shape: (2, 9)
Columns: ['model', 'rows', 'actual_demand_sum', 'wape', 'mae', 'rmse', 'normalized_bias', 'wape_improvement_vs_sn', 'relative_wape_improvement_vs_sn']


model,rows,actual_demand_sum,wape,mae,rmse,normalized_bias,wape_improvement_vs_sn,relative_wape_improvement_vs_sn
XGBoost,24948,11444795.174966,0.187725,86.117867,271.996339,0.120925,0.044378,0.191199
Seasonal Naive,24948,11444795.174966,0.232102,106.475958,380.031724,0.095004,0.000000,0.000000



SOURCE: Historical evaluation context
Path: docs/assets/global_forecasting/final_holdout_historical_context.csv
Shape: (7, 2)
Columns: ['metric', 'value']


metric,value
Historical deployment count,15.000000
Historical mean fold WAPE,0.132503
Historical median fold WAPE,0.122108
Historical std fold WAPE,0.023207
Historical best fold WAPE,0.110548
Historical worst fold WAPE,0.188239
Final untouched holdout WAPE,0.187725


### 2.6 Historical and Final Evaluation Evidence

Forecast accuracy must be interpreted across both historical development periods and the final holdout.

The selected XGBoost training procedure was evaluated across 15 historical rolling deployments:

| Historical metric | Value |
|---|---:|
| Mean fold WAPE | 13.25% |
| Median fold WAPE | 12.21% |
| Best fold WAPE | 11.05% |
| Worst fold WAPE | 18.82% |

On the final August 2–15 holdout, XGBoost achieved 18.77% WAPE, compared with 23.21% for Seasonal Naive on the same observations.

This represents an approximately 19.12% relative reduction in WAPE on that specific holdout.

#### Interpretation

The final August holdout was substantially more challenging than the average historical development window. Its WAPE was close to the worst historical fold.

Consequently, the final holdout should not be presented as the model's universal or typical accuracy.

Conversely, the historical mean should not be treated as a guaranteed expected error for future planning periods. The rolling development results were also used during model development and selection.

The historical Seasonal Naive benchmark used six forecast origins, whereas the XGBoost historical analysis used 15 development deployments. Their aggregate scores should not be interpreted as a directly paired model comparison.

#### Knowledge Base implications

The Model Card should distinguish:

- Historical performance distribution.
- Final holdout performance.
- Comparisons evaluated on identical observations.
- Known variation across evaluation periods.
- Limitations when generalizing historical results to future operations.

The assistant's default answer should communicate the business meaning concisely, without requiring the planner to understand WAPE or temporal cross-validation.

In [5]:
import numpy as np
import pandas as pd

historical = (
    evaluation_tables["Historical evaluation context"]
    .set_index("metric")["value"]
)

holdout = (
    evaluation_tables["Final holdout model comparison"]
    .set_index("model")
)

xgb_holdout = float(holdout.loc["XGBoost", "wape"])
naive_holdout = float(holdout.loc["Seasonal Naive", "wape"])
historical_mean = float(historical["Historical mean fold WAPE"])

relative_improvement = (
    naive_holdout - xgb_holdout
) / naive_holdout

# Verify consistency with the validated model contract.
assert np.isclose(
    xgb_holdout,
    sources["contract"]["final_holdout_metrics"]["wape"],
    atol=1e-10,
)

assert np.isclose(
    xgb_holdout,
    float(historical["Final untouched holdout WAPE"]),
    atol=1e-10,
)

performance_summary = pd.DataFrame([
    {
        "Evaluation scope": "Historical development",
        "Metric": "XGBoost mean fold WAPE",
        "Value": f"{historical_mean:.2%}",
    },
    {
        "Evaluation scope": "Historical development",
        "Metric": "XGBoost median fold WAPE",
        "Value": f'{historical["Historical median fold WAPE"]:.2%}',
    },
    {
        "Evaluation scope": "Final holdout",
        "Metric": "XGBoost WAPE",
        "Value": f"{xgb_holdout:.2%}",
    },
    {
        "Evaluation scope": "Final holdout",
        "Metric": "Seasonal Naive WAPE",
        "Value": f"{naive_holdout:.2%}",
    },
    {
        "Evaluation scope": "Final holdout",
        "Metric": "Relative WAPE reduction",
        "Value": f"{relative_improvement:.2%}",
    },
])

print("K02 — VERIFIED FORECAST ACCURACY EVIDENCE")
display(performance_summary.style.hide(axis="index"))

print(
    "\nHistorical rolling deployments:",
    int(historical["Historical deployment count"]),
)
print(
    "Final holdout evaluated rows:",
    int(holdout.loc["XGBoost", "rows"]),
)

K02 — VERIFIED FORECAST ACCURACY EVIDENCE


Evaluation scope,Metric,Value
Historical development,XGBoost mean fold WAPE,13.25%
Historical development,XGBoost median fold WAPE,12.21%
Final holdout,XGBoost WAPE,18.77%
Final holdout,Seasonal Naive WAPE,23.21%
Final holdout,Relative WAPE reduction,19.12%



Historical rolling deployments: 15
Final holdout evaluated rows: 24948


## 3. Curated Knowledge Base Construction

### 3.1 Building the first knowledge artifact: Model Card

The source discovery and validation stages established a consistent set of model facts across the model contract, cloud evaluation report and historical evaluation artifacts.

We now transform these technical findings into knowledge that can support business-facing questions.

The Model Card will have two layers:

- **Business-facing guidance:** concise explanations of what the forecasting system does, how it can help and what limitations matter for planning decisions.
- **Technical evidence and provenance:** validated metrics, model version, evaluation scope and references to the original artifacts.

The business-facing layer should not require familiarity with statistics or machine learning.

The technical evidence layer preserves precision and traceability without overwhelming the default user experience.

We will create the document as a versioned project artifact, independently of notebook cell positions.

The model contract remains the source of truth for quantitative model properties. The Model Card is a curated interpretation of those properties, tied to a specific model version.

In [6]:
from pathlib import Path
import subprocess
import pandas as pd

contract = sources["contract"]

historical = (
    evaluation_tables["Historical evaluation context"]
    .set_index("metric")["value"]
)

comparison = (
    evaluation_tables["Final holdout model comparison"]
    .set_index("model")
)

model_version = contract["artifact"]["model_version"]
historical_error = float(historical["Historical mean fold WAPE"])
holdout_error = float(comparison.loc["XGBoost", "wape"])
holdout_bias = float(comparison.loc["XGBoost", "normalized_bias"])

assert model_version == "point_forecast_v1"
assert not any("promo" in feature.lower() for feature in contract["model_features"])

source_revision = subprocess.run(
    ["git", "rev-parse", "d2d5a9c20773"],
    cwd=project_root,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

model_card = f"""# Retail Forecasting — Model Card

**Knowledge ID:** `forecast.model_card.v1`
**Model version:** `{model_version}`
**Status:** Selected frozen forecasting champion
**Audience:** Retail planners and forecasting operations

## What does the system do?

It estimates daily sales for each store and product family over the next 14 days.

These forecasts provide a starting point for inventory and replenishment planning.

## How accurate are our forecasts?

Across 15 historical development tests, the average overall forecasting error was around {historical_error:.0%} of sales.

In a particularly challenging August test period, the overall error was closer to {holdout_error:.0%}.

Forecasts are useful planning estimates, not guaranteed sales quantities.

## Can I trust every store and product forecast equally?

No. Forecast accuracy varies across stores, product families and time periods.

A good overall result does not guarantee equally accurate predictions for every individual product family.

## Does the system take promotions into account?

The currently selected forecasting model does not use promotion features.

Promotion-aware approaches were investigated separately but were not retained as the forecasting champion.

## Does low sales always mean low customer demand?

Not necessarily.

The available dataset records sales but does not provide inventory availability or confirmed stockout information.

Low sales could therefore reflect limited product availability rather than low customer demand.

## Can the forecasts determine exactly how much inventory to order?

No. They estimate future sales, but do not establish current stock levels, supplier constraints or optimal order quantities.

Retail planners must combine forecasts with the relevant operational information.

---

## Technical Evidence

| Property | Validated value |
|---|---|
| Model | Global direct multi-horizon XGBoost |
| Champion ID | `{contract["champion_id"]}` |
| Training policy | `{contract["training_policy"]}` |
| Forecast horizon | {contract["forecast_horizon_days"]} days |
| Historical development windows | {int(historical["Historical deployment count"])} |
| Historical mean fold WAPE | {historical_error:.4%} |
| Final holdout WAPE | {holdout_error:.4%} |
| Final holdout normalized bias | {holdout_bias:+.4%} |
| Final holdout period | {contract["final_holdout_start"]} to {contract["final_holdout_end"]} |

Historical development results and the final holdout represent different evaluation scopes.

The historical average is not a guarantee of future performance. The final holdout represents one particular historical period.

These metrics describe aggregate performance, not the accuracy of every individual store-family series.

The model is a validated historical forecasting artifact. This document does not establish that automated daily production forecasting is currently enabled.

## Provenance

Source Git revision: `{source_revision}`

- `artifacts/global_xgboost/point_forecast_model_contract_v1.json`
- `artifacts/aws/sagemaker_registry/champion_v1/evaluation_report_cloud.json`
- `docs/assets/global_forecasting/final_holdout_historical_context.csv`
- `docs/assets/global_forecasting/final_holdout_model_comparison.csv`
- `README.md`

The quantitative fields above are snapshots of the referenced model version. A future champion requires a separately reviewed Model Card.
"""

knowledge_dir = project_root / "docs" / "knowledge"
knowledge_dir.mkdir(parents=True, exist_ok=True)

model_card_path = knowledge_dir / "model_card.md"

if model_card_path.exists():
    assert model_card_path.read_text(encoding="utf-8") == model_card, (
        "The existing Model Card differs from the notebook definition."
    )
else:
    model_card_path.write_text(model_card, encoding="utf-8")

print("MODEL CARD CREATED")
print("Path:", model_card_path.relative_to(project_root))
print("Model version:", model_version)
print("Source revision:", source_revision[:12])
print("Characters:", len(model_card))
print("Sections:", model_card.count("\n## "))

MODEL CARD CREATED
Path: docs/knowledge/model_card.md
Model version: point_forecast_v1
Source revision: d2d5a9c20773
Characters: 3200
Sections: 8


### 3.2 Business Glossary and Forecast Methodology

The Model Card establishes what the forecasting system does and summarizes its validated performance.

We now create two complementary knowledge documents.

**Business Glossary**

Defines recurring retail-planning concepts using language accessible to non-technical users.

**Forecast Methodology**

Explains how forecasts should be interpreted, what decisions they can support and which limitations matter for day-to-day planning.

These documents serve different purposes.

The glossary defines terminology. The methodology explains how to use and interpret the information.

Both documents are designed for business-facing retrieval, while the Model Card and original model artifacts preserve the technical evidence.

The initial versions are curated drafts. Their factual content will be checked against the existing project sources before being approved for knowledge retrieval.

In [7]:
business_glossary = """# Retail Forecasting — Business Glossary

**Knowledge ID:** `forecast.business_glossary.v1`
**Audience:** Retail planners
**Status:** Draft — pending source review

## Sales forecast

An estimate of how much a store is expected to sell over a future period.

A forecast helps anticipate demand, but it is not a guaranteed sales quantity.

## Product family

A group of related products tracked together for forecasting.

Our current system forecasts at product-family level, not individual-product level.

## Forecast horizon

The period covered by a forecast.

Our system estimates daily sales for the next 14 days.

## Forecast accuracy

How closely forecasts match actual sales.

Accuracy can vary between stores, product families and periods. Good overall results do not guarantee equally reliable forecasts everywhere.

## Overforecasting

Predicting more sales than actually occur.

Persistent overforecasting may contribute to excess inventory if purchasing decisions rely too heavily on these predictions.

## Underforecasting

Predicting fewer sales than actually occur.

Persistent underforecasting may increase stockout risk if replenishment decisions do not account for unexpected demand.

## Stockout

A situation in which a product is unavailable for customers to purchase.

Observed sales may underestimate customer demand during stockouts.

## Observed sales versus customer demand

Sales measure what customers actually purchased.

Customer demand may be higher when products are unavailable. Our dataset does not provide the inventory information needed to measure this difference reliably.

## Promotion

A commercial action intended to encourage purchases, such as a discount or special offer.

The currently selected forecasting model does not use promotion features. Promotion effects were investigated separately.

## Replenishment planning

The process of deciding when and how much inventory to restock.

Sales forecasts can support replenishment planning, but inventory levels, supplier lead times and other operational constraints must also be considered.

---

## Sources and Scope

- `README.md`
- `artifacts/global_xgboost/point_forecast_model_contract_v1.json`
- `docs/knowledge/model_card.md`

Definitions describe the current forecasting use case and are not universal claims about all retail forecasting systems.
"""

glossary_path = knowledge_dir / "business_glossary.md"

# Preserve a previously generated document during notebook replay.
# The final version will be validated after the documented revisions.
if not glossary_path.exists():
    glossary_path.write_text(business_glossary, encoding="utf-8")

print("Business Glossary:", glossary_path.relative_to(project_root))

Business Glossary: docs/knowledge/business_glossary.md


In [8]:
forecast_methodology = """# Retail Forecasting — Planning Methodology

**Knowledge ID:** `forecast.planning_methodology.v1`
**Audience:** Retail planners and forecasting operations
**Status:** Draft — pending source review
**Model scope:** `point_forecast_v1`

## What the Forecast Represents

The forecasting system estimates daily sales over the next 14 days for each store and product family.

These estimates help planners anticipate upcoming sales activity.

They are not inventory recommendations, confirmed customer orders or guaranteed sales volumes.

## How to Use Forecasts in Planning

Forecasts can support decisions about where future sales activity may be higher or lower.

However, a replenishment decision also requires information such as:

- Current inventory and incoming stock.
- Supplier lead times and delivery constraints.
- Product availability and operational requirements.

The current forecasting model does not account for all these factors.

Therefore, forecasts should support planning decisions rather than replace operational judgment.

## How Reliable Are the Forecasts?

Historical evaluation provides evidence that the model can produce useful sales estimates.

However, forecast accuracy varies across time periods, stores and product families.

A strong overall result does not guarantee accurate predictions for every individual series.

The Model Card contains the validated evaluation results and their scope.

## Understanding Forecasting Errors

Forecasts may be higher or lower than actual sales.

Repeated overestimation can contribute to unnecessary inventory if it influences replenishment decisions.

Repeated underestimation can increase the risk of insufficient stock.

These are potential planning consequences, not observed inventory outcomes in this dataset.

## Data Availability Limitations

The dataset contains observed sales but does not provide complete inventory availability or confirmed stockout information.

Consequently, the system cannot always distinguish low customer demand from sales constrained by product availability.

Forecasts estimate observed sales patterns rather than unconstrained customer demand.

## Promotions

The selected forecasting model does not use promotion features.

Separate promotion-aware experiments were conducted, but the promotion-aware extension was not retained as the forecasting champion.

The current forecasts should not be interpreted as validated estimates of incremental promotional impact.

## Practical Interpretation

A forecast should answer a practical planning question first.

For example, when a planner asks whether forecasts are reliable, the initial explanation should describe their usefulness and variability in plain business language.

Technical metrics, model architecture and statistical details can be provided when requested.

Important limitations should be mentioned when they materially affect a planning decision, without overwhelming the user with unnecessary details.

---

## Sources and Scope

- `README.md`
- `artifacts/global_xgboost/point_forecast_model_contract_v1.json`
- `docs/knowledge/model_card.md`
- `docs/assets/global_forecasting/final_holdout_historical_context.csv`
- `docs/assets/global_forecasting/final_holdout_model_comparison.csv`

This document describes how to interpret the current forecasting system. It does not establish that automated replenishment decisions or production inventory integrations are implemented.
"""

methodology_path = knowledge_dir / "forecast_methodology.md"

if not methodology_path.exists():
    methodology_path.write_text(forecast_methodology, encoding="utf-8")

print("Forecast Methodology:", methodology_path.relative_to(project_root))

Forecast Methodology: docs/knowledge/forecast_methodology.md


#### Initial Curated-Document Inventory

The first Knowledge Base draft contains a Model Card, a Business Glossary and a Forecast Methodology. The initial inventory below records **these three definitions as first authored**, before the additional technical documents and before the terminology improvements introduced in Section 4.

The notebook is intentionally iterative: subsequent experiments identify retrieval failures and refine the glossary and methodology. On later executions, the Markdown files on disk may already contain those refinements. Reading the current files at this stage would distort the historical baseline; therefore, this initial inventory uses the draft strings defined above.

The final disk-backed versions are validated separately in Section 5.4.

In [9]:
import re
import pandas as pd

# Snapshot of the three initial document definitions in this notebook.
# Files on disk may already contain the improvements made in Section 4.
initial_knowledge_documents = {
    "model_card.md": model_card,
    "business_glossary.md": business_glossary,
    "forecast_methodology.md": forecast_methodology,
}

knowledge_rows = []

for filename, content in initial_knowledge_documents.items():
    match = re.search(
        r"\*\*Knowledge ID:\*\*\s*`([^`]+)`",
        content,
    )

    knowledge_rows.append({
        "document": filename,
        "knowledge_id": match.group(1) if match else None,
        "sections": len(re.findall(r"(?m)^## ", content)),
        "words": len(content.split()),
        "has_sources": (
            "## Sources and Scope" in content
            or "## Provenance" in content
        ),
    })

knowledge_inventory = pd.DataFrame(knowledge_rows)

assert len(knowledge_inventory) == 3
assert knowledge_inventory["knowledge_id"].notna().all()
assert knowledge_inventory["knowledge_id"].is_unique
assert knowledge_inventory["has_sources"].all()

print("INITIAL CURATED KNOWLEDGE INVENTORY")
display(knowledge_inventory.style.hide(axis="index"))

INITIAL CURATED KNOWLEDGE INVENTORY


document,knowledge_id,sections,words,has_sources
model_card.md,forecast.model_card.v1,8,420,True
business_glossary.md,forecast.business_glossary.v1,11,315,True
forecast_methodology.md,forecast.planning_methodology.v1,8,427,True


### 3.3 Feature Dictionary and Operational Runbook

The remaining two planned knowledge artifacts serve primarily technical and operational users.

**Feature Dictionary**

Documents the exact input variables of the selected forecasting model, including their meaning, construction, temporal availability and code provenance.

Feature definitions are derived from the frozen model contract and the Python feature-engineering implementation, rather than inferred solely from column names.

**Operational Runbook**

Documents how to interpret the current system state, investigate common issues and distinguish validated functionality from components that are not yet fully automated.

The runbook must not confuse model registration, successful batch inference or continuous-training completion with automatic production deployment.

Both documents remain version-specific and are treated as reviewable project artifacts.

In [10]:
from pathlib import Path

contract = sources["contract"]
feature_names = contract["model_features"]

feature_definitions = {}

def add_feature(name, group, description, source):
    assert name not in feature_definitions
    feature_definitions[name] = (group, description, source)

origin_source = "src/features/origin.py"
horizon_source = "src/features/horizon.py"
seasonal_source = "src/features/target_aligned.py"

add_feature("store_nbr", "Identity", "Store identifier.", origin_source)
add_feature("family", "Identity", "Product-family identifier.", origin_source)
add_feature("horizon", "Forecast calendar", "Number of days after the forecast origin, from 1 to 14.", horizon_source)

add_feature("sales_at_origin", "Recent sales", "Observed sales on the forecast-origin date T.", origin_source)

for lag in (1, 2, 3, 7, 14, 21, 28, 35, 364):
    add_feature(
        f"sales_lag_{lag}d",
        "Recent sales",
        f"Observed sales {lag} calendar days before the forecast origin T.",
        origin_source,
    )

for window in (7, 14, 28):
    add_feature(
        f"rolling_mean_{window}d",
        "Recent sales",
        f"Mean observed daily sales over the last {window} calendar days, including T; missing observations are ignored.",
        origin_source,
    )

add_feature(
    "recent_level_ratio", "Recent sales",
    "Ratio of the 7-day to the 28-day rolling mean; undefined when the 28-day mean is unavailable or near zero.",
    origin_source,
)
add_feature(
    "rolling_std_28d", "Demand variability",
    "Population standard deviation of observed sales over the last 28 calendar days, including T; at least two observations required.",
    origin_source,
)
add_feature(
    "positive_rate_28d", "Demand intermittency",
    "Fraction of observed days with strictly positive sales in the last 28 calendar days.",
    origin_source,
)
add_feature(
    "positive_sales_mean_28d", "Demand intermittency",
    "Average sales conditional on strictly positive sales within the last 28 calendar days.",
    origin_source,
)
add_feature(
    "days_since_last_positive", "Demand intermittency",
    "Calendar days since the most recent strictly positive observed sales, measured at T.",
    origin_source,
)

add_feature(
    "target_day_of_week", "Forecast calendar",
    "Weekday of the predicted date; Monday is 0 and Sunday is 6.",
    horizon_source,
)
add_feature(
    "target_month", "Forecast calendar",
    "Calendar month of the predicted date, from 1 to 12.",
    horizon_source,
)
add_feature(
    "target_is_weekend", "Forecast calendar",
    "Indicator equal to 1 when the predicted date is Saturday or Sunday.",
    horizon_source,
)

for lag in (364, 728):
    add_feature(
        f"target_sales_lag_{lag}d",
        "Historical seasonality",
        f"Observed sales at the target date shifted backward by {lag} calendar days.",
        seasonal_source,
    )

add_feature(
    "target_sales_lag_1y_calendar", "Historical seasonality",
    "Observed sales at the target date shifted backward by one calendar year.",
    seasonal_source,
)
add_feature(
    "historical_seasonal_level", "Historical seasonality",
    "Mean of available historical 28-day sales levels at up to four annual seasonal positions; each usable window requires at least 27 observed days.",
    seasonal_source,
)
add_feature(
    "seasonal_level_trend_ratio", "Historical seasonality",
    "Linear trend across available annual seasonal levels, divided by their mean; requires at least two usable years and a non-negligible mean.",
    seasonal_source,
)
add_feature(
    "seasonal_years_available", "Historical seasonality",
    "Number of usable historical annual seasonal levels, between 0 and 4.",
    seasonal_source,
)
add_feature(
    "current_vs_seasonal_ratio", "Historical seasonality",
    "Current 28-day rolling sales mean divided by the historical seasonal level, when the denominator is available and nonzero.",
    "src/forecasting/build_forecast.py",
)

assert len(feature_names) == 31
assert len(feature_names) == len(set(feature_names))
assert set(feature_names) == set(feature_definitions), (
    "Feature Dictionary does not match the frozen model contract."
)

feature_rows = []
for name in feature_names:
    group, description, source = feature_definitions[name]
    feature_rows.append({
        "Feature": name,
        "Group": group,
        "Definition": description,
        "Source": source,
    })

feature_dictionary_table = pd.DataFrame(feature_rows)

feature_markdown = feature_dictionary_table.to_markdown(index=False)

feature_dictionary = f"""# Retail Forecasting — Feature Dictionary

**Knowledge ID:** `forecast.feature_dictionary.v1`
**Model version:** `{contract["artifact"]["model_version"]}`
**Status:** Draft — pending review
**Audience:** ML engineering and forecasting operations

## Feature Contract

The frozen forecasting model uses exactly 31 input variables.

All historical sales-based features must be constructed using data available no later than forecast origin T. Calendar properties of future prediction dates are known in advance.

Missing historical observations are not automatically equivalent to zero sales.

## Frozen Model Features

{feature_markdown}

## Important Constraints

- Forecasts are generated for the 14 days following origin T.
- The feature order is defined by the frozen model contract.
- Future realized sales must never be used as predictors.
- Promotion information is not part of the selected V1 predictor schema.
- Feature availability and categorical compatibility must be validated during inference.
- This dictionary describes the V1 model, not all experimental features explored during research.

## Provenance

Model contract:

- `artifacts/global_xgboost/point_forecast_model_contract_v1.json`

Implementation:

- `src/features/origin.py`
- `src/features/horizon.py`
- `src/features/target_aligned.py`
- `src/forecasting/build_forecast.py`
- `src/forecasting/model_inputs.py`

Source Git revision: `{source_revision}`

Feature descriptions are derived from the frozen implementation and require review if the implementation or model version changes.
"""

feature_dictionary_path = knowledge_dir / "feature_dictionary.md"

if feature_dictionary_path.exists():
    assert feature_dictionary_path.read_text(encoding="utf-8") == feature_dictionary
else:
    feature_dictionary_path.write_text(feature_dictionary, encoding="utf-8")

print("FEATURE DICTIONARY CREATED")
print("Path:", feature_dictionary_path.relative_to(project_root))
print("Features documented:", len(feature_dictionary_table))
print("Contract coverage:", "100%")

FEATURE DICTIONARY CREATED
Path: docs/knowledge/feature_dictionary.md
Features documented: 31
Contract coverage: 100%


In [11]:
import json

checkpoint_path = (
    project_root
    / "artifacts/aws/continuous_training/last_successful_ct.json"
)

with checkpoint_path.open("r", encoding="utf-8") as file:
    ct_checkpoint = json.load(file)

assert ct_checkpoint["schema_version"] == "ct_checkpoint_v1"
assert ct_checkpoint["last_cycle_promoted"] is False
assert ct_checkpoint["last_cycle_new_out_of_sample_wape_available"] is False
assert ct_checkpoint["last_challenger_approval_status"] == "PendingManualApproval"

operations_runbook = f"""# Retail Forecasting — Operational Runbook

**Knowledge ID:** `forecast.operations_runbook.v1`
**Status:** Draft — validated historical project state
**Audience:** Forecasting operations and ML engineering

## System Scope

The project includes AWS-based model evaluation, batch inference, SageMaker Model Registry and guarded continuous-training workflows.

The validated forecasting champion is `{contract["artifact"]["model_version"]}`.

This runbook documents demonstrated capabilities and known constraints. It is not evidence of an active, fully automated production service.

## 1. Investigating Forecast Quality

When forecast quality is questioned:

1. Identify the model version and forecast origin.
2. Verify the forecast dates and store-family coverage.
3. Locate the matching forecasts and realized sales.
4. Use evaluation reports for the same model and observation period.
5. Check both overall forecasting error and systematic overestimation or underestimation.
6. Avoid extrapolating portfolio-wide accuracy to an individual store-family series.

Primary references:

- `artifacts/global_xgboost/point_forecast_model_contract_v1.json`
- `artifacts/aws/sagemaker_registry/champion_v1/evaluation_report_cloud.json`

## 2. Investigating Data and Feature Issues

Check that:

- Sales data contains the required store, family, date and sales fields.
- Store-family-date keys are unique.
- Historical observations do not extend beyond the forecast origin.
- Missing historical sales are not silently treated as observed zero sales.
- The exact frozen model feature contract is respected.
- The inference input does not contain future target values.

Primary references:

- `src/features/origin.py`
- `src/features/horizon.py`
- `src/features/target_aligned.py`
- `src/forecasting/model_inputs.py`

## 3. Continuous-Training State

The recorded checkpoint has the following historical state:

- Completed cutoff: `{ct_checkpoint["last_successful_ct_cutoff"]}`
- Completed phase: `{ct_checkpoint["last_successful_cycle_phase"]}`
- Challenger approval: `{ct_checkpoint["last_challenger_approval_status"]}`
- Challenger promoted: `{ct_checkpoint["last_cycle_promoted"]}`
- New out-of-sample performance available: `{ct_checkpoint["last_cycle_new_out_of_sample_wape_available"]}`

The successful CT cycle produced a registered challenger awaiting manual approval.

It did not automatically promote a new forecasting model.

The latest validated CT-6C readiness workflow does not automatically execute the entire next training cycle.

## 4. Safe Operating Principles

- Do not automatically approve or promote a challenger based only on successful training or registration.
- Do not claim improved forecasting accuracy without appropriate evaluation evidence.
- Treat a blocked readiness gate as a reason to investigate the missing requirements, not as an instruction to bypass safeguards.
- Use versioned evaluation evidence when communicating model performance.
- Distinguish historical project execution from current service availability.

## 5. Known Operational Limitations

The project does not establish a fully automated production retraining and deployment lifecycle.

The forecasting system does not provide live inventory data or automatic replenishment decisions.

The operational investigation agent and its production-facing interfaces are developed separately.

## Provenance

- `README.md`
- `artifacts/aws/continuous_training/last_successful_ct.json`
- `pipelines/processing/ct_workflow_readiness.py`
- `artifacts/aws/sagemaker_registry/champion_v1/registry_record.json`

Source Git revision: `{source_revision}`

This runbook describes the recorded project state, not a live AWS health check or a replacement for deployment-specific incident procedures.
"""

runbook_path = knowledge_dir / "operations_runbook.md"

if runbook_path.exists():
    assert runbook_path.read_text(encoding="utf-8") == operations_runbook
else:
    runbook_path.write_text(operations_runbook, encoding="utf-8")

print("OPERATIONAL RUNBOOK CREATED")
print("Path:", runbook_path.relative_to(project_root))
print("Checkpoint phase:", ct_checkpoint["last_successful_cycle_phase"])

OPERATIONAL RUNBOOK CREATED
Path: docs/knowledge/operations_runbook.md
Checkpoint phase: CHALLENGER_REGISTERED_PENDING


## 4. Knowledge Access and Retrieval

### 4.1 Selecting the Appropriate Access Method

The initial Knowledge Base now contains five curated documents, each with a distinct purpose.

However, not every question should trigger the same retrieval mechanism.

We distinguish two complementary access methods:

**Deterministic structured lookup**

Used for validated numerical facts, model configuration and other fields with a defined machine-readable source of truth.

**Document retrieval**

Used for business explanations, assumptions and limitations that require natural-language interpretation.

The default document-retrieval corpus contains:

- `model_card.md`
- `business_glossary.md`
- `forecast_methodology.md`

The Feature Dictionary and Operational Runbook are retained for technical and operational queries rather than included in every business-facing retrieval.

### 4.2 Retrieval Baseline

We start with a simple lexical retrieval baseline using TF-IDF and cosine similarity.

The curated documents are divided into sections using their existing Markdown headings.

Unlike the earlier research-notebook approach, these sections were deliberately authored around distinct concepts and user needs.

Each retrieved passage retains:

- A stable knowledge-document identifier.
- A semantic section identifier.
- Its source file.
- Its original content.

This initial approach requires no embeddings, vector database or cloud infrastructure.

We will evaluate it before deciding whether more complex retrieval is justified.

#### 4.2.1 Reproducible Experimental Corpus

To preserve a meaningful *before → after* comparison, the first TF-IDF index is built from the **initial document definitions in memory**, rather than from whatever versions happen to exist on disk at execution time.

This matters because the first retrieval experiments revealed terminology mismatches. Later sections improve the **Overforecasting / Underforecasting** definitions and the **Promotions** explanation. If we indexed the already-refined disk files here, the original failures would disappear and the development story could no longer be reproduced.

Each experiment updates the in-memory document snapshot and rebuilds the index. The improved versions are also saved as the final knowledge documents, which the reusable production-oriented module reads from disk in Section 5.

The first baseline and the subsequent revisions are therefore reproducible even after the notebook has been run previously.

In [12]:
import hashlib
import re
import pandas as pd

draft_documents = {
    "model_card.md": model_card,
    "business_glossary.md": business_glossary,
    "forecast_methodology.md": forecast_methodology,
}

business_document_names = [
    "model_card.md",
    "business_glossary.md",
    "forecast_methodology.md",
]

excluded_sections = {
    "Technical Evidence",
    "Provenance",
    "Sources and Scope",
}

def make_section_slug(title):
    return re.sub(
        r"[^a-z0-9]+", "-",
        title.lower()
    ).strip("-")

business_passages = []

for filename in business_document_names:
    path = knowledge_dir / filename
    content = draft_documents[filename]

    id_match = re.search(
        r"\*\*Knowledge ID:\*\*\s*`([^`]+)`",
        content,
    )
    assert id_match, f"Missing Knowledge ID: {filename}"

    knowledge_id = id_match.group(1)
    headings = list(re.finditer(r"(?m)^## (.+)$", content))

    for index, heading in enumerate(headings):
        title = heading.group(1).strip()

        if title in excluded_sections:
            continue

        end = (
            headings[index + 1].start()
            if index + 1 < len(headings)
            else len(content)
        )

        passage = content[heading.start():end].strip()

        business_passages.append({
            "passage_id": (
                f"{knowledge_id}::{make_section_slug(title)}"
            ),
            "knowledge_id": knowledge_id,
            "document": filename,
            "section": title,
            "text": passage,
            "source_sha256": hashlib.sha256(
                content.encode("utf-8")
            ).hexdigest(),
        })

passage_ids = [p["passage_id"] for p in business_passages]

assert len(passage_ids) == len(set(passage_ids))
assert all(p["text"] for p in business_passages)

corpus_summary = (
    pd.DataFrame(business_passages)
    .groupby("document")
    .agg(
        passages=("passage_id", "count"),
        mean_words=("text", lambda x: round(
            x.str.split().str.len().mean(), 1
        )),
    )
    .reset_index()
)

print("BUSINESS KNOWLEDGE CORPUS")
display(corpus_summary.style.hide(axis="index"))

print("Total passages:", len(business_passages))

BUSINESS KNOWLEDGE CORPUS


document,passages,mean_words
business_glossary.md,10,27.000000
forecast_methodology.md,7,52.100000
model_card.md,6,37.000000


Total passages: 23


In [13]:
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

business_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True,
)

business_corpus_matrix = business_vectorizer.fit_transform(
    [passage["text"] for passage in business_passages]
)

def search_business_knowledge(question, top_k=3):
    """Retrieve relevant curated business explanations."""
    query_vector = business_vectorizer.transform([question])

    scores = cosine_similarity(
        query_vector, business_corpus_matrix
    ).ravel()

    ranked_indices = np.argsort(-scores, kind="stable")
    results = []

    for index in ranked_indices[:top_k]:
        if scores[index] <= 0:
            continue

        passage = business_passages[index]

        results.append({
            **passage,
            "score": float(scores[index]),
        })

    return results

test_questions = [
    "Can I trust forecasts for every store and product?",
    "Does low sales mean customers don't want the product?",
    "What happens if the forecasts are too high?",
    "Does the system account for promotions?",
    "How should I use forecasts when planning stock?",
]

retrieval_rows = []

for question in test_questions:
    results = search_business_knowledge(question, top_k=3)

    for rank, result in enumerate(results, start=1):
        retrieval_rows.append({
            "Question": question,
            "Rank": rank,
            "Document": result["document"],
            "Section": result["section"],
            "Score": round(result["score"], 3),
        })

retrieval_results = pd.DataFrame(retrieval_rows)

print("CURATED KNOWLEDGE — INITIAL RETRIEVAL")
display(retrieval_results.style.hide(axis="index"))

CURATED KNOWLEDGE — INITIAL RETRIEVAL


Question,Rank,Document,Section,Score
Can I trust forecasts for every store and product?,1,model_card.md,Can I trust every store and product forecast equally?,0.286000
Can I trust forecasts for every store and product?,2,model_card.md,What does the system do?,0.223000
Can I trust forecasts for every store and product?,3,forecast_methodology.md,What the Forecast Represents,0.143000
Does low sales mean customers don't want the product?,1,model_card.md,Does low sales always mean low customer demand?,0.451000
Does low sales mean customers don't want the product?,2,business_glossary.md,Stockout,0.111000
Does low sales mean customers don't want the product?,3,forecast_methodology.md,Data Availability Limitations,0.091000
What happens if the forecasts are too high?,1,business_glossary.md,Forecast accuracy,0.143000
What happens if the forecasts are too high?,2,model_card.md,Can the forecasts determine exactly how much inventory to order?,0.123000
What happens if the forecasts are too high?,3,forecast_methodology.md,How to Use Forecasts in Planning,0.120000
Does the system account for promotions?,1,model_card.md,Does the system take promotions into account?,0.218000


### 4.3 Deterministic Access to Validated Model Facts

Certain business questions require precise numerical information rather than document retrieval.

Examples include:

- Which forecasting model is selected?
- How far ahead does it predict?
- What was its historical forecasting error?
- What was its error during the final evaluation period?

These values are already available in validated structured artifacts.

A deterministic lookup avoids searching for numerical facts in prose and reduces the risk of confusing different model versions or evaluation periods.

The following function provides a minimal read-only interface for these facts.

Natural-language interpretation and tool selection will be developed in the subsequent GenAI and Agentic AI notebooks.

In [14]:
def get_validated_forecast_facts(topic):
    """Return version-specific facts from validated project artifacts."""
    contract = sources["contract"]

    historical = (
        evaluation_tables["Historical evaluation context"]
        .set_index("metric")["value"]
    )

    if topic == "model":
        return {
            "champion_id": contract["champion_id"],
            "model_version": contract["artifact"]["model_version"],
            "forecast_horizon_days": contract["forecast_horizon_days"],
            "training_policy": contract["training_policy"],
            "source": (
                "artifacts/global_xgboost/"
                "point_forecast_model_contract_v1.json"
            ),
        }

    if topic == "accuracy":
        return {
            "model_version": contract["artifact"]["model_version"],
            "historical_development_windows": int(
                historical["Historical deployment count"]
            ),
            "historical_mean_fold_wape": float(
                historical["Historical mean fold WAPE"]
            ),
            "final_holdout_wape": float(
                contract["final_holdout_metrics"]["wape"]
            ),
            "final_holdout_start": contract["final_holdout_start"],
            "final_holdout_end": contract["final_holdout_end"],
            "scope_note": (
                "Historical development average and final holdout "
                "refer to different evaluation scopes."
            ),
            "sources": [
                "artifacts/global_xgboost/"
                "point_forecast_model_contract_v1.json",
                "docs/assets/global_forecasting/"
                "final_holdout_historical_context.csv",
            ],
        }

    raise ValueError(f"Unsupported fact topic: {topic}")

model_facts = get_validated_forecast_facts("model")
accuracy_facts = get_validated_forecast_facts("accuracy")

assert model_facts["forecast_horizon_days"] == 14
assert 0 < accuracy_facts["historical_mean_fold_wape"] < 1
assert 0 < accuracy_facts["final_holdout_wape"] < 1

print("STRUCTURED KNOWLEDGE LOOKUP")
print("Champion:", model_facts["model_version"])
print("Forecast horizon:", model_facts["forecast_horizon_days"], "days")
print(
    "Historical development error:",
    f'{accuracy_facts["historical_mean_fold_wape"]:.2%}',
)
print(
    "Final holdout error:",
    f'{accuracy_facts["final_holdout_wape"]:.2%}',
)

STRUCTURED KNOWLEDGE LOOKUP
Champion: point_forecast_v1
Forecast horizon: 14 days
Historical development error: 13.25%
Final holdout error: 18.77%


In [15]:
retrieval_test_cases = [
    {
        "question": "Can I trust forecasts for every store and product?",
        "expected_document": "model_card.md",
        "expected_section": "Can I trust every store and product forecast equally?",
    },
    {
        "question": "Does low sales always mean low customer demand?",
        "expected_document": "business_glossary.md",
        "expected_section": "Observed sales versus customer demand",
    },
    {
        "question": "What happens when forecasts are too high?",
        "expected_document": "business_glossary.md",
        "expected_section": "Overforecasting",
    },
    {
        "question": "Does the system use promotions?",
        "expected_document": "forecast_methodology.md",
        "expected_section": "Promotions",
    },
    {
        "question": "How should I use the forecast for stock planning?",
        "expected_document": "forecast_methodology.md",
        "expected_section": "How to Use Forecasts in Planning",
    },
]

evaluation_rows = []

for case in retrieval_test_cases:
    results = search_business_knowledge(
        case["question"], top_k=3
    )

    expected_found = any(
        result["document"] == case["expected_document"]
        and result["section"] == case["expected_section"]
        for result in results
    )

    evaluation_rows.append({
        "Question": case["question"],
        "Expected section": case["expected_section"],
        "Retrieved rank 1": (
            results[0]["section"] if results else None
        ),
        "Expected in top 3": expected_found,
    })

retrieval_evaluation = pd.DataFrame(evaluation_rows)

print("CURATED RETRIEVAL — INITIAL EVALUATION")
display(retrieval_evaluation.style.hide(axis="index"))

print(
    "Exact expected-section hits:",
    int(retrieval_evaluation["Expected in top 3"].sum()),
    "/",
    len(retrieval_evaluation),
)

CURATED RETRIEVAL — INITIAL EVALUATION


Question,Expected section,Retrieved rank 1,Expected in top 3
Can I trust forecasts for every store and product?,Can I trust every store and product forecast equally?,Can I trust every store and product forecast equally?,True
Does low sales always mean low customer demand?,Observed sales versus customer demand,Does low sales always mean low customer demand?,True
What happens when forecasts are too high?,Overforecasting,Forecast accuracy,False
Does the system use promotions?,Promotions,Does the system take promotions into account?,True
How should I use the forecast for stock planning?,How to Use Forecasts in Planning,How to Use Forecasts in Planning,True


Exact expected-section hits: 4 / 5


### 4.4 Initial Retrieval Evaluation

The first curated retrieval baseline indexed 23 business-oriented passages from three documents.

A five-question smoke test retrieved the predefined expected section within the top three results for four questions.

Manual inspection revealed that several questions were already answered adequately by a different section than the one specified in the test.

The main observed failure concerned a natural-language question about forecasts being too high.

The relevant glossary section used the technical term "overforecasting" without explicitly including the planner's everyday wording.

**Decision:** improve the business terminology of the source document before introducing more complex retrieval techniques.

This is a targeted knowledge-curation correction, not evidence that lexical retrieval is universally sufficient.

The smoke test is small and was derived from the curated documentation. Its results should not be interpreted as an independent estimate of production retrieval quality.

In [16]:
revisions = {
    "Predicting more sales than actually occur.": (
        "When forecasts are too high, the system predicts "
        "more sales than actually occur."
    ),
    "Predicting fewer sales than actually occur.": (
        "When forecasts are too low, the system predicts "
        "fewer sales than actually occur."
    ),
}

# Apply the experimental revision to the original draft.
revised_glossary = business_glossary

for original, revised in revisions.items():
    if original in revised_glossary:
        revised_glossary = revised_glossary.replace(original, revised, 1)
    else:
        assert revised in revised_glossary

# Accept either the original or the already revised file.
# Never silently overwrite unrelated manual changes.
current_glossary = glossary_path.read_text(encoding="utf-8")

assert current_glossary in (business_glossary, revised_glossary), (
    "Unexpected changes in the existing Business Glossary."
)

if current_glossary != revised_glossary:
    glossary_path.write_text(revised_glossary, encoding="utf-8")

business_glossary = revised_glossary
draft_documents["business_glossary.md"] = business_glossary

# Rebuild the business passage index using the existing section policy.
business_passages = []

for filename in business_document_names:
    path = knowledge_dir / filename
    content = draft_documents[filename]

    id_match = re.search(
        r"\*\*Knowledge ID:\*\*\s*`([^`]+)`",
        content,
    )
    assert id_match

    knowledge_id = id_match.group(1)
    headings = list(re.finditer(r"(?m)^## (.+)$", content))

    for index, heading in enumerate(headings):
        title = heading.group(1).strip()
        if title in excluded_sections:
            continue

        end = (
            headings[index + 1].start()
            if index + 1 < len(headings)
            else len(content)
        )

        business_passages.append({
            "passage_id": f"{knowledge_id}::{make_section_slug(title)}",
            "knowledge_id": knowledge_id,
            "document": filename,
            "section": title,
            "text": content[heading.start():end].strip(),
            "source_sha256": hashlib.sha256(
                content.encode("utf-8")
            ).hexdigest(),
        })

business_corpus_matrix = business_vectorizer.fit_transform(
    [passage["text"] for passage in business_passages]
)

assert len(business_passages) == 23
assert len({
    passage["passage_id"] for passage in business_passages
}) == 23

print("Curated passages reindexed:", len(business_passages))

Curated passages reindexed: 23


In [17]:
updated_evaluation_rows = []

for case in retrieval_test_cases:
    results = search_business_knowledge(case["question"], top_k=3)

    expected_rank = next(
        (
            rank
            for rank, result in enumerate(results, start=1)
            if result["document"] == case["expected_document"]
            and result["section"] == case["expected_section"]
        ),
        None,
    )

    updated_evaluation_rows.append({
        "Question": case["question"],
        "Top 1 section": results[0]["section"] if results else None,
        "Expected rank": expected_rank,
        "Expected in Top 3": expected_rank is not None,
    })

updated_evaluation = pd.DataFrame(updated_evaluation_rows)

print("CURATED RETRIEVAL — AFTER BUSINESS TERMINOLOGY REVISION")
display(updated_evaluation.style.hide(axis="index"))

print(
    "\nExpected-section hits:",
    int(updated_evaluation["Expected in Top 3"].sum()),
    "/",
    len(updated_evaluation),
)

print("\nFORECASTS TOO HIGH — RETRIEVED EVIDENCE")
for rank, result in enumerate(
    search_business_knowledge(
        "What happens when forecasts are too high?",
        top_k=3,
    ),
    start=1,
):
    print(
        f"{rank}. {result['document']} / {result['section']}"
        f" (score={result['score']:.3f})"
    )

CURATED RETRIEVAL — AFTER BUSINESS TERMINOLOGY REVISION


Question,Top 1 section,Expected rank,Expected in Top 3
Can I trust forecasts for every store and product?,Can I trust every store and product forecast equally?,1,True
Does low sales always mean low customer demand?,Does low sales always mean low customer demand?,3,True
What happens when forecasts are too high?,Overforecasting,1,True
Does the system use promotions?,Does the system take promotions into account?,2,True
How should I use the forecast for stock planning?,How to Use Forecasts in Planning,1,True



Expected-section hits: 5 / 5

FORECASTS TOO HIGH — RETRIEVED EVIDENCE
1. business_glossary.md / Overforecasting (score=0.275)
2. business_glossary.md / Forecast accuracy (score=0.037)
3. model_card.md / Can the forecasts determine exactly how much inventory to order? (score=0.032)


### 4.5 Business-Oriented Retrieval Validation

The first retrieval smoke test achieved 4/5 expected-section hits at rank 3.

After updating the business glossary to explicitly recognize everyday expressions such as "forecasts are too high", the result improved to 5/5.

This confirms that business-oriented documentation can improve lexical retrieval without requiring additional model complexity.

However, the original questions were used during development and cannot establish general retrieval performance.

#### Additional business-oriented validation

We now examine additional questions phrased from the perspective of a retail planner.

The objective is to verify whether retrieved passages contain sufficient evidence to support a concise, non-technical answer.

We also distinguish three information-access requirements:

- **DOCUMENT:** explanations already available in the curated business knowledge.
- **STRUCTURED:** validated numerical facts requiring deterministic lookup.
- **OPERATIONAL:** questions requiring forecast data or operational tools not available through the documentary corpus.

This evaluation checks evidence selection, not LLM answer generation.

The expected source type is specified manually as part of the test design. Automatic question routing is outside the scope of this notebook.

#### Acceptance criteria

We will favor a simple retrieval approach if:

1. Business questions retrieve sufficient supporting evidence without misleading context.
2. Quantitative model facts remain accessible through structured lookup.
3. Operational questions are explicitly recognized as requiring dedicated tools.
4. The approach remains straightforward to maintain and integrate.

These are development checks, not evidence of production-ready accuracy.

In [18]:
business_validation_cases = [
    {
        "id": "V01",
        "question": "Are these predictions for individual products or product groups?",
        "access": "DOCUMENT",
        "required_evidence": "Forecasts are produced at product-family level, not individual SKU level.",
    },
    {
        "id": "V02",
        "question": "If a store sells nothing today, does that mean nobody wanted to buy?",
        "access": "DOCUMENT",
        "required_evidence": "Observed sales may be constrained by missing product availability.",
    },
    {
        "id": "V03",
        "question": "Could relying on these predictions leave us with too much stock?",
        "access": "DOCUMENT",
        "required_evidence": "Overforecasting can contribute to excess inventory.",
    },
    {
        "id": "V04",
        "question": "Will tomorrow's estimates reflect our planned discounts?",
        "access": "DOCUMENT",
        "required_evidence": "The current forecasting champion does not use promotion features.",
    },
    {
        "id": "V05",
        "question": "Can I use these numbers as my final purchase order quantities?",
        "access": "DOCUMENT",
        "required_evidence": "Forecasts alone do not account for inventory, supplier and replenishment constraints.",
    },
    {
        "id": "V06",
        "question": "Are the estimates equally dependable across all our stores?",
        "access": "DOCUMENT",
        "required_evidence": "Forecast quality varies across stores, product families and time periods.",
    },
    {
        "id": "V07",
        "question": "How much overall forecasting error did we see in historical testing?",
        "access": "STRUCTURED",
        "required_evidence": "Historical mean fold WAPE from the validated historical evaluation artifact.",
    },
    {
        "id": "V08",
        "question": "How far into the future can the system predict?",
        "access": "STRUCTURED",
        "required_evidence": "The model contract specifies a 14-day forecast horizon.",
    },
    {
        "id": "V09",
        "question": "How many units should store 12 expect to sell tomorrow?",
        "access": "OPERATIONAL",
        "required_evidence": "Store-specific forecast data, including the product family and forecast origin.",
    },
    {
        "id": "V10",
        "question": "When was the most recent forecast actually generated?",
        "access": "OPERATIONAL",
        "required_evidence": "Current execution metadata from the forecasting pipeline.",
    },
]

validation_rows = []

for case in business_validation_cases:
    row = {
        "ID": case["id"],
        "Question": case["question"],
        "Required access": case["access"],
        "Required evidence": case["required_evidence"],
        "Rank 1": None,
        "Rank 2": None,
    }

    if case["access"] == "DOCUMENT":
        results = search_business_knowledge(
            case["question"], top_k=2
        )

        for rank, result in enumerate(results, start=1):
            row[f"Rank {rank}"] = (
                f'{result["document"]} / {result["section"]}'
            )

    validation_rows.append(row)

business_validation = pd.DataFrame(validation_rows)

print("BUSINESS-ORIENTED KNOWLEDGE VALIDATION")
display(business_validation.style.hide(axis="index"))

BUSINESS-ORIENTED KNOWLEDGE VALIDATION


ID,Question,Required access,Required evidence,Rank 1,Rank 2
V01,Are these predictions for individual products or product groups?,DOCUMENT,"Forecasts are produced at product-family level, not individual SKU level.",model_card.md / Can I trust every store and product forecast equally?,business_glossary.md / Product family
V02,"If a store sells nothing today, does that mean nobody wanted to buy?",DOCUMENT,Observed sales may be constrained by missing product availability.,model_card.md / Does low sales always mean low customer demand?,model_card.md / What does the system do?
V03,Could relying on these predictions leave us with too much stock?,DOCUMENT,Overforecasting can contribute to excess inventory.,business_glossary.md / Overforecasting,model_card.md / Can I trust every store and product forecast equally?
V04,Will tomorrow's estimates reflect our planned discounts?,DOCUMENT,The current forecasting champion does not use promotion features.,model_card.md / Does low sales always mean low customer demand?,forecast_methodology.md / What the Forecast Represents
V05,Can I use these numbers as my final purchase order quantities?,DOCUMENT,"Forecasts alone do not account for inventory, supplier and replenishment constraints.",model_card.md / Can the forecasts determine exactly how much inventory to order?,business_glossary.md / Stockout
V06,Are the estimates equally dependable across all our stores?,DOCUMENT,"Forecast quality varies across stores, product families and time periods.",model_card.md / Can I trust every store and product forecast equally?,business_glossary.md / Forecast accuracy
V07,How much overall forecasting error did we see in historical testing?,STRUCTURED,Historical mean fold WAPE from the validated historical evaluation artifact.,nan,nan
V08,How far into the future can the system predict?,STRUCTURED,The model contract specifies a 14-day forecast horizon.,nan,nan
V09,How many units should store 12 expect to sell tomorrow?,OPERATIONAL,"Store-specific forecast data, including the product family and forecast origin.",nan,nan
V10,When was the most recent forecast actually generated?,OPERATIONAL,Current execution metadata from the forecasting pipeline.,nan,nan


In [19]:
evidence_rows = []

for case in business_validation_cases:
    if case["access"] != "DOCUMENT":
        continue

    results = search_business_knowledge(
        case["question"], top_k=2
    )

    for rank, result in enumerate(results, start=1):
        evidence_rows.append({
            "case_id": case["id"],
            "question": case["question"],
            "rank": rank,
            "passage_id": result["passage_id"],
            "score": round(result["score"], 3),
            "retrieved_evidence": result["text"],
            "required_evidence": case["required_evidence"],
        })

business_evidence_review = pd.DataFrame(evidence_rows)

print("DOCUMENT EVIDENCE — REVIEW TABLE")
display(
    business_evidence_review.assign(
        preview=lambda df: df["retrieved_evidence"].str[:250]
        .str.replace("\n", " ", regex=False)
    )[[
        "case_id", "rank", "passage_id",
        "score", "preview",
    ]].style.hide(axis="index")
)

print("\nSTRUCTURED LOOKUP CHECK")
facts = get_validated_forecast_facts("accuracy")
model = get_validated_forecast_facts("model")

print("Historical average error:", f'{facts["historical_mean_fold_wape"]:.2%}')
print("Forecast horizon:", model["forecast_horizon_days"], "days")

assert len(business_evidence_review) == 12
assert model["forecast_horizon_days"] == 14

DOCUMENT EVIDENCE — REVIEW TABLE


case_id,rank,passage_id,score,preview
V01,1,forecast.model_card.v1::can-i-trust-every-store-and-product-forecast-equally,0.253000,"## Can I trust every store and product forecast equally? No. Forecast accuracy varies across stores, product families and time periods. A good overall result does not guarantee equally accurate predictions for every individual product family."
V01,2,forecast.business_glossary.v1::product-family,0.227000,"## Product family A group of related products tracked together for forecasting. Our current system forecasts at product-family level, not individual-product level."
V02,1,forecast.model_card.v1::does-low-sales-always-mean-low-customer-demand,0.161000,## Does low sales always mean low customer demand? Not necessarily. The available dataset records sales but does not provide inventory availability or confirmed stockout information. Low sales could therefore reflect limited product availability r
V02,2,forecast.model_card.v1::what-does-the-system-do,0.131000,## What does the system do? It estimates daily sales for each store and product family over the next 14 days. These forecasts provide a starting point for inventory and replenishment planning.
V03,1,forecast.business_glossary.v1::overforecasting,0.106000,"## Overforecasting When forecasts are too high, the system predicts more sales than actually occur. Persistent overforecasting may contribute to excess inventory if purchasing decisions rely too heavily on these predictions."
V03,2,forecast.model_card.v1::can-i-trust-every-store-and-product-forecast-equally,0.097000,"## Can I trust every store and product forecast equally? No. Forecast accuracy varies across stores, product families and time periods. A good overall result does not guarantee equally accurate predictions for every individual product family."
V04,1,forecast.model_card.v1::does-low-sales-always-mean-low-customer-demand,0.127000,## Does low sales always mean low customer demand? Not necessarily. The available dataset records sales but does not provide inventory availability or confirmed stockout information. Low sales could therefore reflect limited product availability r
V04,2,forecast.planning_methodology.v1::what-the-forecast-represents,0.094000,"## What the Forecast Represents The forecasting system estimates daily sales over the next 14 days for each store and product family. These estimates help planners anticipate upcoming sales activity. They are not inventory recommendations, confirm"
V05,1,forecast.model_card.v1::can-the-forecasts-determine-exactly-how-much-inventory-to-order,0.262000,"## Can the forecasts determine exactly how much inventory to order? No. They estimate future sales, but do not establish current stock levels, supplier constraints or optimal order quantities. Retail planners must combine forecasts with the relevan"
V05,2,forecast.business_glossary.v1::stockout,0.113000,## Stockout A situation in which a product is unavailable for customers to purchase. Observed sales may underestimate customer demand during stockouts.



STRUCTURED LOOKUP CHECK
Historical average error: 13.25%
Forecast horizon: 14 days


#### 4.5.1 Addressing a Business-Language Retrieval Gap

The additional business-oriented evaluation revealed a retrieval failure for question V04:

> "Will tomorrow's estimates reflect our planned discounts?"

The first two retrieved passages discussed sales and forecasts but did not explain how promotions are handled.

The relevant information was already present in the Knowledge Base, under the **Promotions** section of the Forecast Methodology.

The problem was therefore not missing knowledge. It was a mismatch between the language used by the planner ("planned discounts") and the terminology used in our documentation ("promotion features").

This matters because retail planners should not need to know the system's technical vocabulary to find relevant information.

**Decision:** refine the existing business documentation to explicitly connect planned discounts with promotions, rather than introducing additional retrieval complexity.

The revised wording also preserves an important distinction: the selected model does not explicitly use planned promotions as input features, although historical sales patterns may indirectly reflect past promotional activity.

We then rebuild the TF-IDF index and repeat the affected retrieval test.

This illustrates an important knowledge-engineering principle: retrieval quality depends not only on the search algorithm, but also on how clearly the underlying knowledge is written for its intended users.

In [20]:
old_phrase = "The selected forecasting model does not use promotion features."
new_phrase = (
    "The selected forecasting model does not explicitly use "
    "planned discounts or other promotion information as input features."
)

def apply_document_revision(content):
    if new_phrase in content:
        return content
    assert content.count(old_phrase) == 1, "Unexpected methodology content"
    return content.replace(old_phrase, new_phrase, 1)

forecast_methodology = apply_document_revision(forecast_methodology)

current_methodology = methodology_path.read_text(encoding="utf-8")
updated_methodology = apply_document_revision(current_methodology)

if updated_methodology != current_methodology:
    methodology_path.write_text(updated_methodology, encoding="utf-8")

assert methodology_path.read_text(encoding="utf-8") == forecast_methodology

draft_documents["forecast_methodology.md"] = forecast_methodology

print("Methodology updated and synchronized with notebook definition.")

Methodology updated and synchronized with notebook definition.


In [21]:
def load_business_passages():
    passages = []

    for filename in business_document_names:
        path = knowledge_dir / filename
        content = path.read_text(encoding="utf-8")

        id_match = re.search(
            r"\*\*Knowledge ID:\*\*\s*`([^`]+)`",
            content,
        )
        assert id_match, f"Missing Knowledge ID: {filename}"

        knowledge_id = id_match.group(1)
        headings = list(re.finditer(r"(?m)^## (.+)$", content))
        document_hash = hashlib.sha256(
            content.encode("utf-8")
        ).hexdigest()

        for index, heading in enumerate(headings):
            title = heading.group(1).strip()

            if title in excluded_sections:
                continue

            end = (
                headings[index + 1].start()
                if index + 1 < len(headings)
                else len(content)
            )

            passages.append({
                "passage_id": f"{knowledge_id}::{make_section_slug(title)}",
                "knowledge_id": knowledge_id,
                "document": filename,
                "section": title,
                "text": content[heading.start():end].strip(),
                "source_sha256": document_hash,
            })

    assert len({
        passage["passage_id"] for passage in passages
    }) == len(passages)

    return passages

business_passages = load_business_passages()

business_corpus_matrix = business_vectorizer.fit_transform(
    [passage["text"] for passage in business_passages]
)

question = "Will tomorrow's estimates reflect our planned discounts?"
results = search_business_knowledge(question, top_k=3)

print("V04 — PLANNED DISCOUNTS")
for rank, result in enumerate(results, start=1):
    print(
        f'{rank}. {result["document"]} / {result["section"]}'
        f' | score={result["score"]:.3f}'
    )

print("\nCORPUS SIZE:", len(business_passages))

V04 — PLANNED DISCOUNTS
1. forecast_methodology.md / Promotions | score=0.220
2. model_card.md / Does low sales always mean low customer demand? | score=0.072
3. forecast_methodology.md / What the Forecast Represents | score=0.053

CORPUS SIZE: 23


#### 4.5.2 Evaluation Result and Decision

After the terminology revision, the **Promotions** section became the top-ranked result for V04, with a cosine similarity score of 0.220.

The business corpus still contains 23 passages.

The improvement was achieved by refining the source documentation, without changing the retrieval algorithm or introducing additional infrastructure.

**Decision:** retain the curated TF-IDF baseline for the initial knowledge-access implementation.

The evaluation also shows that lexical retrieval can remain sensitive to the wording of a question. Its results should be treated as supporting evidence, not as a guarantee that the retrieved passages fully answer every question.

The next step is to make this retrieval capability reusable outside the notebook.

### 4.6 Knowledge Access Contract

The curated Knowledge Base provides two complementary capabilities:

1. **Document retrieval:** returns business explanations with stable passage identifiers and source provenance.
2. **Structured fact lookup:** returns validated numerical information directly from versioned artifacts.

The initial TF-IDF retrieval baseline operates on a small, business-oriented corpus.

The first five development questions retrieved their expected sections within the top three results after a terminology correction.

An additional six-question business validation identified one important terminology mismatch involving planned discounts. This led to a targeted documentation revision.

These observations support retaining TF-IDF as a lightweight initial retrieval strategy, but do not establish general retrieval accuracy.

#### Interface requirements

The downstream GenAI system must receive:

- The original user question.
- Retrieved evidence with stable identifiers.
- Document provenance and version information.
- The complete evidence text.
- Explicitly retrieved structured facts when applicable.

Retrieval scores are used for ranking, not as calibrated probabilities of factual correctness.

The retrieval layer must not invent missing information or automatically answer questions requiring operational data.

Question routing, natural-language generation, tool orchestration and user-adaptive explanations will be handled in subsequent notebooks.

The Knowledge Base is currently a local, curated development artifact. Its documents still require final review before they can be treated as approved production knowledge.

In [22]:
def build_document_context(question, top_k=3):
    """Build a provenance-aware evidence payload for grounded generation."""
    matches = search_business_knowledge(question, top_k=top_k)

    evidence = []

    for rank, match in enumerate(matches, start=1):
        source_path = knowledge_dir / match["document"]
        current_hash = hashlib.sha256(
            source_path.read_bytes()
        ).hexdigest()

        if current_hash != match["source_sha256"]:
            raise RuntimeError(
                f"Knowledge source changed after indexing: {match['document']}"
            )

        evidence.append({
            "rank": rank,
            "passage_id": match["passage_id"],
            "knowledge_id": match["knowledge_id"],
            "source_path": str(
                source_path.relative_to(project_root)
            ),
            "source_sha256": current_hash,
            "section": match["section"],
            "content": match["text"],
        })

    return {
        "question": question,
        "retrieval_method": "curated_tfidf_v1",
        "evidence": evidence,
    }

# Demonstrate the interface on a business question.
example_context = build_document_context(
    "Could relying on these forecasts leave us with too much stock?",
    top_k=2,
)

print("KNOWLEDGE ACCESS CONTRACT")
print("Question:", example_context["question"])
print("Retrieval method:", example_context["retrieval_method"])
print("Evidence count:", len(example_context["evidence"]))

for item in example_context["evidence"]:
    print(f'\n[{item["rank"]}] {item["passage_id"]}')
    print("Source:", item["source_path"])
    print("Content:", item["content"])

KNOWLEDGE ACCESS CONTRACT
Question: Could relying on these forecasts leave us with too much stock?
Retrieval method: curated_tfidf_v1
Evidence count: 2

[1] forecast.model_card.v1::can-the-forecasts-determine-exactly-how-much-inventory-to-order
Source: docs/knowledge/model_card.md
Content: ## Can the forecasts determine exactly how much inventory to order?

No. They estimate future sales, but do not establish current stock levels, supplier constraints or optimal order quantities.

Retail planners must combine forecasts with the relevant operational information.

---

[2] forecast.planning_methodology.v1::how-to-use-forecasts-in-planning
Source: docs/knowledge/forecast_methodology.md
Content: ## How to Use Forecasts in Planning

Forecasts can support decisions about where future sales activity may be higher or lower.

However, a replenishment decision also requires information such as:

- Current inventory and incoming stock.
- Supplier lead times and delivery constraints.
- Product ava

## 5. Reusable Knowledge Access Component

### 5.1 Moving from Notebook Experimentation to Reusable Code

The previous sections established a curated business corpus, a lexical retrieval baseline and a provenance-aware evidence format.

However, the implementation still depends on variables and functions defined inside this notebook.

This is unsuitable for reuse by other components.

We therefore move the document-loading, section-indexing and retrieval logic into a small Python module under `src/knowledge/`.

The notebook remains responsible for documenting the methodology, running experiments and validating the component.

The reusable module will:

- Load the curated business documents directly from the project repository.
- Generate stable passage identifiers from knowledge IDs and section headings.
- Build a TF-IDF index over business-facing passages.
- Return ranked passages with their source metadata.
- Detect document changes after indexing to prevent stale evidence from being silently reused.

The initial implementation is deliberately local and read-only.

It does not require AWS services, an embedding model or a vector database.

Grounded answer generation and automatic tool selection are developed separately.

In [23]:
import sys
import importlib

src_dir = (project_root / "src").resolve()

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import knowledge.retrieval as retrieval_module

# Reload the actual Python module, not just the import paths.
importlib.reload(retrieval_module)

BusinessKnowledgeRetriever = retrieval_module.BusinessKnowledgeRetriever
knowledge_retriever = BusinessKnowledgeRetriever(project_root)

assert len(knowledge_retriever.passages) == 23

print("Module loaded from:", retrieval_module.__file__)
print("Knowledge retriever ready.")
print("Indexed passages:", len(knowledge_retriever.passages))

Module loaded from: /Users/zaamine/Desktop/Amine/Training/Kaggle/Corporacion Favorita/retail-demand-forecasting-platform-aws/src/knowledge/retrieval.py
Knowledge retriever ready.
Indexed passages: 23


In [24]:
import sys

src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from knowledge.retrieval import BusinessKnowledgeRetriever

knowledge_retriever = BusinessKnowledgeRetriever(project_root)

questions = [
    "Are these forecasts reliable for all stores?",
    "Will the forecast account for our planned discounts?",
    "Could we end up ordering too much stock?",
]

integration_rows = []

for question in questions:
    context = knowledge_retriever.build_context(
        question, top_k=3
    )

    integration_rows.append({
        "question": question,
        "evidence_count": len(context["evidence"]),
        "top_section": (
            context["evidence"][0]["section"]
            if context["evidence"] else None
        ),
        "top_source": (
            context["evidence"][0]["source_path"]
            if context["evidence"] else None
        ),
    })

integration_results = pd.DataFrame(integration_rows)

print("REUSABLE KNOWLEDGE RETRIEVER — INTEGRATION TEST")
display(integration_results.style.hide(axis="index"))

assert len(knowledge_retriever.passages) == 23
assert all(
    result["evidence_count"] > 0
    for result in integration_rows
)

REUSABLE KNOWLEDGE RETRIEVER — INTEGRATION TEST


question,evidence_count,top_section,top_source
Are these forecasts reliable for all stores?,3,Forecast accuracy,docs/knowledge/business_glossary.md
Will the forecast account for our planned discounts?,3,Promotions,docs/knowledge/forecast_methodology.md
Could we end up ordering too much stock?,3,Can the forecasts determine exactly how much inventory to order?,docs/knowledge/model_card.md


### 5.2 Reusable Structured Knowledge Access

The document retriever provides explanations and business context. However, validated numerical facts should be read directly from their authoritative structured sources.

We therefore implement a separate read-only interface for model identity and forecasting accuracy.

The interface:

- Reads the frozen model contract and relevant evaluation artifacts.
- Checks consistency between local and cloud evaluation evidence.
- Returns version-specific facts with source provenance.
- Keeps historical development metrics separate from the final holdout.
- Rejects unsupported fact topics.

This separation prevents the future language model from having to extract or infer numerical results from retrieved prose.

The knowledge layer now provides two complementary services: documentary evidence retrieval and deterministic structured fact lookup.

In [25]:
from knowledge.facts import get_validated_forecast_facts

model_facts = get_validated_forecast_facts(
    "model", project_root
)
accuracy_facts = get_validated_forecast_facts(
    "accuracy", project_root
)

document_context = knowledge_retriever.build_context(
    "Will our planned discounts affect tomorrow's forecast?",
    top_k=2,
)

assert model_facts["feature_count"] == 31
assert model_facts["forecast_horizon_days"] == 14
assert accuracy_facts["historical_development_windows"] == 15
assert document_context["evidence"]

print("KNOWLEDGE ACCESS — INTEGRATION")
print("Model:", model_facts["model_version"])
print("Features:", model_facts["feature_count"])
print("Historical WAPE:", f'{accuracy_facts["historical_mean_fold_wape"]:.2%}')
print("Final holdout WAPE:", f'{accuracy_facts["final_holdout_wape"]:.2%}')
print("Retrieved evidence:", document_context["evidence"][0]["section"])

KNOWLEDGE ACCESS — INTEGRATION
Model: point_forecast_v1
Features: 31
Historical WAPE: 13.25%
Final holdout WAPE: 18.77%
Retrieved evidence: Promotions


### 5.3 Integration and Regression Tests

Before integrating the Knowledge Base with a language model, we validate the reusable Python components independently.

The tests cover:

- Correct loading of model metadata and evaluation facts.
- Agreement with the frozen forecasting contract.
- Successful retrieval of relevant business evidence.
- Detection of changes to documents after indexing.

Document-change testing uses temporary copies of the knowledge files, leaving the original project documentation untouched.

These tests validate the software interface and selected invariants. They do not assess the quality of generated language-model answers.

In [26]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_knowledge_access.py",
    ],
    cwd=project_root,
    check=True,
)

print("Knowledge access component validated.")

.....                                                                    [100%]
5 passed in 1.09s
Knowledge access component validated.


### 5.4 Reproducibility and Final Validation

The Knowledge Base is generated from versioned project artifacts and curated documentation definitions.

To preserve reproducibility, the source revision recorded in the knowledge documents refers to the validated forecasting evidence snapshot, rather than the repository's latest commit.

This distinction matters because subsequent changes to notebooks or documentation should not silently alter the provenance of the underlying forecasting model.

The final validation verifies that:

- All five knowledge documents exist.
- Their contents match the definitions maintained in this notebook.
- The reusable retrieval and structured-fact interfaces work independently.
- The automated knowledge-access tests pass.

The knowledge artifacts remain version-specific. Future changes to the forecasting champion or source evidence require explicit review and regeneration of the affected documents.

The knowledge documents are development artifacts, not automatically approved production knowledge.

In [27]:
knowledge_artifacts = {
    model_card_path: model_card,
    glossary_path: business_glossary,
    methodology_path: forecast_methodology,
    feature_dictionary_path: feature_dictionary,
    runbook_path: operations_runbook,
}

for path, expected_content in knowledge_artifacts.items():
    assert path.is_file(), f"Missing knowledge document: {path}"
    assert path.read_text(encoding="utf-8") == expected_content, (
        f"Notebook definition and document differ: {path.name}"
    )

assert len(knowledge_retriever.passages) == 23

print("FINAL KNOWLEDGE BASE VALIDATION")
print("Documents consistent:", len(knowledge_artifacts))
print("Business retrieval passages:", len(knowledge_retriever.passages))
print("Model version:", model_facts["model_version"])

FINAL KNOWLEDGE BASE VALIDATION
Documents consistent: 5
Business retrieval passages: 23
Model version: point_forecast_v1


## 6. Conclusions and GenAI Handoff

### 6.1 Deliverables

This notebook established the first version of the Retail Planner's curated knowledge layer.

The implementation includes:

- Five versioned knowledge documents covering the model, business concepts, forecasting methodology, feature definitions and operational guidance.
- A local TF-IDF retriever over 23 business-oriented sections.
- Stable passage identifiers and source-integrity checks.
- Deterministic access to validated model and evaluation facts.
- Reusable Python components under `src/knowledge/`.
- Integration and regression tests.

### 6.2 Design Decisions

The knowledge layer deliberately distinguishes business explanations, numerical facts and operational information.

The default retrieval corpus prioritizes business-facing documents. Technical documentation remains available separately for more specialized questions.

Business terminology was refined in response to observed retrieval failures involving overforecasting and planned discounts.

The initial TF-IDF approach was retained because it provides a lightweight, interpretable baseline. The current evaluation is not sufficient to establish production retrieval accuracy.

### 6.3 Limitations

The documents remain curated development artifacts requiring final factual and operational review.

The current retriever does not provide automatic question routing, semantic understanding guarantees, live forecast data or operational actions.

Numerical facts describe a validated historical model version, not live forecasting performance.

### 6.4 Next Stage

Notebook 08 will integrate the knowledge-access components with Amazon Bedrock to investigate:

1. Evidence-grounded answer generation.
2. Business-language response quality.
3. Failure handling and answer evaluation.
4. Prompt engineering and fine-tuning feasibility.
5. LLMOps instrumentation and tracing.

Notebook 09 will extend the system with operational tools, agent orchestration and Model Context Protocol (MCP) experiments.

The guiding principle remains unchanged: the Retail Planner should receive direct, accessible answers, with technical detail available when needed.